In [1]:
"""
Contrastive EEG-to-Text Generation Model (BERT-Only Version)
Uses concept pre-training followed by generative fine-tuning
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
# --- We need BertModel (for text encoding) and BertLMHeadModel (for decoding) ---
from transformers import BertModel, BertLMHeadModel
from typing import Optional, Tuple, Dict


class SimplifiedEEGEncoder(nn.Module):
    """
    FIXED: Pure Transformer-based EEG encoder.
    This version correctly treats TIME as the sequence and CHANNELS as the features.
    """
    
    def __init__(
        self, 
        num_channels: int = 62,
        time_steps: int = 400, # This is now the sequence length
        d_model: int = 512,
        nhead: int = 8,
        num_layers: int = 4,
        dropout: float = 0.1
    ):
        super().__init__()
        self.num_channels = num_channels
        self.time_steps = time_steps
        self.d_model = d_model
        
        # Project CHANNELS (features) to d_model
        self.channel_embedding = nn.Linear(num_channels, d_model)
        
        # Positional encoding is over TIME
        self.channel_pos_encoding = nn.Parameter(
            torch.randn(1, time_steps, d_model) * 0.02
        )
        
        # Transformer encoder (runs over the time dimension)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True, # We will pass [batch, 400, 512]
            norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, 
            num_layers=num_layers
        )
        
        self.output_norm = nn.LayerNorm(d_model)
        
        print(f"FIXED SimplifiedEEGEncoder: {num_channels} channels -> {d_model}d, over {time_steps} steps")
    
    def forward(self, eeg: torch.Tensor) -> torch.Tensor:
        """
        Args:
            eeg: [batch, channels(62), time_steps(400)]
        Returns:
            encoded: [batch, time_steps(400), d_model(512)]
        """
        
        # Transpose from [B, C, T] to [B, T, C]
        eeg = eeg.transpose(1, 2) # [batch, 400, 62]
        
        # Project channel features to d_model
        x = self.channel_embedding(eeg) # [batch, 400, 512]
        
        # Add positional encoding
        x = x + self.channel_pos_encoding
        
        # Apply transformer
        x = self.transformer_encoder(x)
        x = self.output_norm(x)
        
        return x


class ProjectionHead(nn.Module):
    """MLP projection head for contrastive learning."""
    
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.LayerNorm(output_dim)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# =========================================================================
# --- THIS IS THE MISSING CLASS ---
# =========================================================================
class EEGToGPT2Adapter(nn.Module):
    """
    Adapter for EEG encoder to a decoder (e.g., BERT or GPT-2).
    (The name EEGToGPT2Adapter is fine, even if we use BERT)
    """
    
    def __init__(
        self,
        eeg_dim: int = 512,
        gpt2_dim: int = 768, # This is the target decoder's dimension
        num_eeg_tokens: int = 8,
        dropout: float = 0.1
    ):
        super().__init__()
        self.num_eeg_tokens = num_eeg_tokens
        
        # Learnable query tokens
        self.query_tokens = nn.Parameter(
            torch.randn(1, num_eeg_tokens, eeg_dim) * 0.02
        )
        
        # Cross-attention (processes sequence length of 400)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=eeg_dim,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        # Project to Decoder's space
        self.projection = nn.Sequential(
            nn.Linear(eeg_dim, gpt2_dim), # Maps 512 -> 768
            nn.LayerNorm(gpt2_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(gpt2_dim, gpt2_dim),
            nn.LayerNorm(gpt2_dim)
        )
        
        print(f"EEGToAdapter: {eeg_dim}d -> {num_eeg_tokens} tokens -> {gpt2_dim}d")
    
    def forward(self, eeg_features: torch.Tensor) -> torch.Tensor:
        # eeg_features is [batch, 400, 512]
        batch_size = eeg_features.shape[0]
        queries = self.query_tokens.expand(batch_size, -1, -1) # [batch, 8, 512]
        
        # Attend to the 400 time steps
        compressed, _ = self.cross_attention(
            query=queries,
            key=eeg_features,
            value=eeg_features
        )
        
        adapted = self.projection(compressed) # [batch, 8, 768]
        return adapted
# =========================================================================


class SigmoidFocalLoss(nn.Module):
    """Focal Loss for multi-label classification."""
    
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


class MetadataHead(nn.Module):
    """Metadata prediction head."""
    
    def __init__(
        self,
        eeg_dim: int = 512,
        num_colors: int = 12,
        num_objects: int = 90,
        dropout: float = 0.1
    ):
        super().__init__()
        
        # Pools over the time dimension (400)
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.color_head = nn.Sequential(
            nn.Linear(eeg_dim, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(dropout),
            nn.Linear(256, num_colors)
        )
        
        self.object_head = nn.Sequential(
            nn.Linear(eeg_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_objects)
        )
    
    def forward(self, eeg_features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # eeg_features is [batch, 400, 512]
        # transpose to [batch, 512, 400] for pooling
        pooled = self.pool(eeg_features.transpose(1, 2)).squeeze(-1) # [batch, 512]
        
        color_logits = self.color_head(pooled)
        object_logits = self.object_head(pooled)
        return color_logits, object_logits


class EEGToTextModel(nn.Module):
    """
    Complete EEG-to-Text model (BERT-Only Version).
    """
    
    def __init__(
        self,
        num_channels: int = 62,
        time_steps: int = 400,
        num_colors: int = 12,
        num_objects: int = 90,
        bert_model_name: str = "bert-base-uncased",
        eeg_encoder_dim: int = 512,
        projection_dim: int = 256,
        num_eeg_prefix_tokens: int = 8,
        dropout: float = 0.1
    ):
        super().__init__()
        
        # EEG Encoder
        self.eeg_encoder = SimplifiedEEGEncoder(
            num_channels=num_channels,
            time_steps=time_steps,
            d_model=eeg_encoder_dim,
            dropout=dropout
        )
        
        # BERT Text Encoder (frozen)
        print(f"Loading BERT from {bert_model_name}...")
        self.text_encoder = BertModel.from_pretrained(bert_model_name)
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        print("BERT text encoder frozen")
        
        bert_dim = self.text_encoder.config.hidden_size
        
        # Projection heads for contrastive learning
        # Pool across time (400) before projecting
        self.eeg_projection = ProjectionHead(
            input_dim=eeg_encoder_dim,
            hidden_dim=512,
            output_dim=projection_dim,
            dropout=dropout
        )
        
        self.text_projection = ProjectionHead(
            input_dim=bert_dim,
            hidden_dim=512,
            output_dim=projection_dim,
            dropout=dropout
        )
        
        print(f"Projection heads: {eeg_encoder_dim}d, {bert_dim}d -> {projection_dim}d")
        
        # BERT Decoder (replaces GPT-2)
        print(f"Loading BERT LM Head from {bert_model_name}...")
        self.decoder = BertLMHeadModel.from_pretrained(bert_model_name)
        self.decoder.config.is_decoder = True
        
        decoder_dim = self.decoder.config.hidden_size
        
        for param in self.decoder.parameters():
            param.requires_grad = False
        print("BERT Decoder weights frozen initially")
        
        # Adapter
        self.adapter = EEGToGPT2Adapter(
            eeg_dim=eeg_encoder_dim,
            gpt2_dim=decoder_dim,
            num_eeg_tokens=num_eeg_prefix_tokens,
            dropout=dropout
        )
        
        # Metadata heads
        self.metadata_head = MetadataHead(
            eeg_dim=eeg_encoder_dim,
            num_colors=num_colors,
            num_objects=num_objects,
            dropout=dropout
        )
        
        self.num_eeg_prefix_tokens = num_eeg_prefix_tokens
        self.projection_dim = projection_dim
        
        # Temperature for contrastive loss
        self.temperature = nn.Parameter(torch.ones([]) * 0.07)
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
    
    def unfreeze_decoder_top_layers(self, num_layers: int = 4):
        """Unfreeze top N transformer layers of BERT Decoder."""
        total_layers = len(self.decoder.bert.encoder.layer)
        for i in range(total_layers - num_layers, total_layers):
            for param in self.decoder.bert.encoder.layer[i].parameters():
                param.requires_grad = True
        
        # Also unfreeze layer norm and LM head
        for param in self.decoder.bert.pooler.parameters():
             param.requires_grad = True
        for param in self.decoder.cls.parameters():
            param.requires_grad = True
        
        print(f"Unfroze top {num_layers} layers of BERT Decoder")
    
    def unfreeze_decoder_all(self):
        """Unfreeze all BERT Decoder layers."""
        for param in self.decoder.parameters():
            param.requires_grad = True
        print("Unfroze all BERT Decoder layers")
    
    def encode_eeg(self, eeg: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        features = self.eeg_encoder(eeg) # [batch, 400, 512]
        
        # Pool for projection (average across time)
        pooled = features.mean(dim=1)  # [batch, 512]
        projection = self.eeg_projection(pooled)
        
        projection = F.normalize(projection, p=2, dim=-1)
        
        return features, projection
    
    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            cls_output = outputs.last_hidden_state[:, 0, :]
        
        projection = self.text_projection(cls_output)
        projection = F.normalize(projection, p=2, dim=-1)
        return projection
    
    def forward(
        self,
        eeg: torch.Tensor,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        text_input_ids: Optional[torch.Tensor] = None,
        text_attention_mask: Optional[torch.Tensor] = None,
        mode: str = 'generate',
        return_metadata: bool = False
    ) -> Dict[str, torch.Tensor]:
        
        result = {}
        eeg_features, eeg_proj = self.encode_eeg(eeg)
        
        if mode == 'contrastive':
            assert text_input_ids is not None, "Need text_input_ids for contrastive mode"
            text_proj = self.encode_text(text_input_ids, text_attention_mask)
            
            result['eeg_projection'] = eeg_proj
            result['text_projection'] = text_proj
            result['temperature'] = self.temperature
        
        if mode in ['generate', 'multitask']:
            assert input_ids is not None, "Need input_ids for generation mode"
            
            eeg_prefix = self.adapter(eeg_features)
            
            text_embeds = self.decoder.bert.embeddings.word_embeddings(input_ids)
            
            inputs_embeds = torch.cat([eeg_prefix, text_embeds], dim=1)
            
            batch_size = eeg.shape[0]
            if attention_mask is None:
                attention_mask = torch.ones(
                    batch_size, input_ids.shape[1],
                    dtype=torch.long, device=input_ids.device
                )
            
            eeg_attention_mask = torch.ones(
                batch_size, self.num_eeg_prefix_tokens,
                dtype=torch.long, device=attention_mask.device
            )
            full_attention_mask = torch.cat([eeg_attention_mask, attention_mask], dim=1)
            
            full_labels = None
            if labels is not None:
                eeg_labels = torch.full(
                    (batch_size, self.num_eeg_prefix_tokens),
                    -100, dtype=torch.long, device=labels.device
                )
                full_labels = torch.cat([eeg_labels, labels], dim=1)
            
            outputs = self.decoder(
                inputs_embeds=inputs_embeds,
                attention_mask=full_attention_mask,
                labels=full_labels,
                return_dict=True
            )
            
            result['logits'] = outputs.logits
            result['text_loss'] = outputs.loss if labels is not None else None
        
        if return_metadata or mode == 'multitask':
            color_logits, object_logits = self.metadata_head(eeg_features)
            result['color_logits'] = color_logits
            result['object_logits'] = object_logits
        
        return result
    
    @torch.no_grad()
    def generate(
        self,
        eeg: torch.Tensor,
        tokenizer,
        max_length: int = 50,
        num_beams: int = 4,
        temperature: float = 1.0,
        top_k: int = 50,
        top_p: float = 0.95,
        repetition_penalty: float = 1.2
    ) -> list:
        self.eval()
        batch_size = eeg.shape[0]
        device = eeg.device
        
        eeg_features, _ = self.encode_eeg(eeg)
        eeg_prefix = self.adapter(eeg_features) # [batch, 8, 768]
        
        start_token_id = tokenizer.cls_token_id
        input_ids = torch.full(
            (batch_size, 1),
            start_token_id,
            dtype=torch.long,
            device=device
        )
        
        start_embeds = self.decoder.bert.embeddings.word_embeddings(input_ids)
        
        inputs_embeds = torch.cat([eeg_prefix, start_embeds], dim=1)
        
        attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=device)
        
        total_max_length = max_length + self.num_eeg_prefix_tokens + 1
        
        pad_token_id = tokenizer.pad_token_id
        eos_token_id = tokenizer.sep_token_id # BERT uses SEP as EOS

        generated_ids = self.decoder.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_length=total_max_length,
            num_beams=num_beams,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=pad_token_id,
            eos_token_id=eos_token_id,
            do_sample=(num_beams == 1 and temperature > 0.0),
            early_stopping=True
        )
        
        generated_ids = generated_ids[:, (self.num_eeg_prefix_tokens + 1):]
        generated_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        
        return generated_texts

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
def compute_metrics_numpy(
    color_preds,
    color_targets,
    object_preds,
    object_targets
):
    """
    Computes metadata metrics using pure NumPy.
    (Adapted from your evaluate.py)
    """
    color_preds = np.array(color_preds)
    color_targets = np.array(color_targets)
    object_preds = np.array(object_preds)
    object_targets = np.array(object_targets)
    
    # Color accuracy
    color_acc = (color_preds == color_targets).mean()
    
    # Object metrics (Micro F1)
    tp = (object_preds * object_targets).sum()
    fp = (object_preds * (1 - object_targets)).sum()
    fn = ((1 - object_preds) * object_targets).sum()
    
    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    f1_micro = 2 * precision * recall / (precision + recall + 1e-10)
    
    return {
        'color_accuracy': float(color_acc),
        'object_f1_micro': float(f1_micro)
    }

In [3]:
"""
Three-Phase Training Strategy with Contrastive Alignment
Phase 1: Contrastive alignment (InfoNCE loss)
Phase 2: Generative fine-tuning (text-only)
Phase 3: Multitask polishing (with focal loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from tqdm.auto import tqdm
import time
import json
from pathlib import Path


class EEGTextDataset(Dataset):
    """Dataset for EEG and text with caching."""
    
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))
        
        return eeg, meta, text


def collate_fn(batch, pad_id):
    """Collate batch with padding."""
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, text in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(text)
    
    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=pad_id)
    
    return eeg_batch, meta_batch, text_padded


def info_nce_loss(
    eeg_proj: torch.Tensor,
    text_proj: torch.Tensor,
    temperature: torch.Tensor
) -> torch.Tensor:
    """
    InfoNCE (NT-Xent) loss for contrastive learning.
    
    Args:
        eeg_proj: [batch, proj_dim] - normalized EEG projections
        text_proj: [batch, proj_dim] - normalized text projections
        temperature: learnable temperature parameter
    
    Returns:
        loss: scalar
    """
    batch_size = eeg_proj.shape[0]
    
    # Compute similarity matrix [batch, batch]
    logits = torch.matmul(eeg_proj, text_proj.T) / temperature
    
    # Labels: diagonal elements are positive pairs
    labels = torch.arange(batch_size, device=logits.device)
    
    # Cross-entropy loss
    # Loss is symmetric: eeg->text + text->eeg
    loss_eeg_to_text = F.cross_entropy(logits, labels)
    loss_text_to_eeg = F.cross_entropy(logits.T, labels)
    
    loss = (loss_eeg_to_text + loss_text_to_eeg) / 2
    
    return loss


# =========================================================================
# --- REVISED ThreePhaseTrainer CLASS ---
# (Replace the old one in Cell 2)
# =========================================================================

# --- Make sure numpy is imported at the top of your cell ---
# import numpy as np

def compute_metrics_numpy(
    color_preds,
    color_targets,
    object_preds,
    object_targets
):
    """
    Computes metadata metrics using pure NumPy.
    (Adapted from your evaluate.py)
    """
    color_preds = np.array(color_preds)
    color_targets = np.array(color_targets)
    object_preds = np.array(object_preds)
    object_targets = np.array(object_targets)
    
    # Color accuracy
    color_acc = (color_preds == color_targets).mean()
    
    # Object metrics (Micro F1)
    tp = (object_preds * object_targets).sum()
    fp = (object_preds * (1 - object_targets)).sum()
    fn = ((1 - object_preds) * object_targets).sum()
    
    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    f1_micro = 2 * precision * recall / (precision + recall + 1e-10)
    
    return {
        'color_accuracy': float(color_acc),
        'object_f1_micro': float(f1_micro)
    }


class ThreePhaseTrainer:
    """
    NEW 3-Phase Trainer (No torchmetrics dependency):
    1. Pre-train Encoder on Metadata (concepts)
    2. Train Adapter + GPT-2 on Text (generation)
    3. Fine-tune all (polishing)
    """
    
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        tokenizer,
        device,
        output_dir="./checkpoints",
        num_colors=12,
        num_objects=90
    ):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.tokenizer = tokenizer
        self.device = device
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.num_colors = num_colors
        self.num_objects = num_objects
        
        # Loss functions
        self.text_criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
        self.color_criterion = nn.CrossEntropyLoss()
        self.object_criterion = SigmoidFocalLoss(alpha=0.25, gamma=2.0)
        
        self.history = {
            'phase1': [], 'phase2': [], 'phase3': []
        }

    # =========================================================================
    # --- PHASE 1 (NEW): Train Encoder on Metadata ---
    # =========================================================================
    def phase1_concept_pretraining(self, epochs=15, lr=1e-3, warmup_steps=500):
        print("\n" + "="*80)
        print("PHASE 1: Concept Pre-training (Training EEG-Encoder + Metadata-Heads)")
        print("="*80)
        
        # --- FIX: Define new loss weights ---
        COLOR_WEIGHT = 1.0
        OBJECT_WEIGHT = 100.0
        print(f"Phase 1 Loss Weights: Color={COLOR_WEIGHT}, Object={OBJECT_WEIGHT}")
        
        params_to_optimize = list(self.model.eeg_encoder.parameters()) + \
                           list(self.model.metadata_head.parameters())
        
        optimizer = torch.optim.AdamW(params_to_optimize, lr=lr, weight_decay=0.01)
        
        total_steps = len(self.train_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
        )
        
        best_val_loss = float('inf')
        
        for epoch in range(1, epochs + 1):
            print(f"\nEpoch {epoch}/{epochs}")
            start_time = time.time()
            
            # --- Train ---
            self.model.train()
            total_loss = 0.0
            total_color_loss = 0.0
            total_object_loss = 0.0
            
            pbar = tqdm(self.train_loader, desc=f"Phase1 Epoch {epoch}")
            for eeg, metadata, _ in pbar:
                eeg = eeg.to(self.device)
                metadata = metadata.to(self.device)
                
                optimizer.zero_grad()
                
                eeg_features, _ = self.model.encode_eeg(eeg)
                color_logits, object_logits = self.model.metadata_head(eeg_features)
                
                color_targets = metadata[:, 0].long()
                color_loss = self.color_criterion(color_logits, color_targets)
                
                object_targets = metadata[:, 1:].float()
                object_loss = self.object_criterion(object_logits, object_targets)
                
                # --- FIX: Apply new weights ---
                loss = (COLOR_WEIGHT * color_loss) + (OBJECT_WEIGHT * object_loss)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(params_to_optimize, 1.0)
                optimizer.step()
                scheduler.step()
                
                total_loss += loss.item()
                total_color_loss += color_loss.item()
                total_object_loss += object_loss.item()
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'clr': f'{color_loss.item():.4f}',
                    'obj': f'{object_loss.item():.4f}'
                })
            
            # --- Validate ---
            val_metrics = self._validate_metadata(
                color_weight=COLOR_WEIGHT, object_weight=OBJECT_WEIGHT
            )
            val_loss = val_metrics['loss']
            
            elapsed = time.time() - start_time
            n = len(self.train_loader)
            
            print(f"Train - Loss: {total_loss/n:.4f} | "
                  f"Color: {total_color_loss/n:.4f} | "
                  f"Object: {total_object_loss/n:.4f}")
            
            print(f"Val - Loss: {val_metrics['loss']:.4f} | "
                  f"Color Acc: {val_metrics['color_accuracy']:.4f} (Baseline: ~0.083) | "
                  f"Object F1: {val_metrics['object_f1_micro']:.4f} (Baseline: 0.0)")
            
            print(f"Time: {elapsed:.1f}s")
            
            self.history['phase1'].append({
                'epoch': epoch,
                'train_loss': total_loss / n,
                'val_loss': val_loss,
                'val_color_acc': val_metrics['color_accuracy'],
                'val_object_f1': val_metrics['object_f1_micro'],
            })
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(self.model.state_dict(), self.output_dir / 'phase1_best.pt')
                print(f"✓ Saved best Phase 1 model (val_loss={best_val_loss:.4f})")
        
        self.model.load_state_dict(torch.load(self.output_dir / 'phase1_best.pt'))
        print("\nPhase 1 complete! EEG Encoder is pre-trained on concepts.")
    
    
    @torch.no_grad()
    def _validate_metadata(self, color_weight, object_weight):
        """Validate metadata loss AND METRICS using NumPy."""
        self.model.eval()
        total_loss = 0.0
        total_color_loss = 0.0
        total_object_loss = 0.0
        
        all_color_preds = []
        all_color_targets = []
        all_object_preds = []
        all_object_targets = []
        
        for eeg, metadata, _ in self.val_loader:
            eeg = eeg.to(self.device)
            metadata = metadata.to(self.device)
            
            eeg_features, _ = self.model.encode_eeg(eeg)
            color_logits, object_logits = self.model.metadata_head(eeg_features)
            
            color_targets = metadata[:, 0].long()
            color_loss = self.color_criterion(color_logits, color_targets)
            
            object_targets = metadata[:, 1:].float()
            object_loss = self.object_criterion(object_logits, object_targets)
            
            # --- FIX: Apply new weights ---
            loss = (color_weight * color_loss) + (object_weight * object_loss)
            
            total_loss += loss.item()
            total_color_loss += color_loss.item()
            total_object_loss += object_loss.item()
            
            all_color_preds.append(color_logits.argmax(dim=-1).cpu())
            all_color_targets.append(color_targets.cpu())
            all_object_preds.append((torch.sigmoid(object_logits) > 0.5).cpu())
            all_object_targets.append(object_targets.cpu())
        
        all_color_preds = np.concatenate(all_color_preds)
        all_color_targets = np.concatenate(all_color_targets)
        all_object_preds = np.concatenate(all_object_preds)
        all_object_targets = np.concatenate(all_object_targets)
        
        metrics = compute_metrics_numpy(
            all_color_preds,
            all_color_targets,
            all_object_preds,
            all_object_targets
        )
        
        n = len(self.val_loader)
        return {
            'loss': total_loss / n,
            'color_loss': total_color_loss / n,
            'object_loss': total_object_loss / n,
            'color_accuracy': metrics['color_accuracy'],
            'object_f1_micro': metrics['object_f1_micro']
        }

    #
    # --- NO CHANGES NEEDED FOR PHASE 2 OR PHASE 3 ---
    #
    
    def phase2_generative_finetuning(self, epochs=10, lr=1e-4, warmup_steps=300):
        print("\n" + "="*80)
        print("PHASE 2: Generative Fine-tuning (Training Adapter + Top BERT)")
        print("="*80)
        
        # Freeze EEG encoder and metadata head (they are our experts now)
        for param in self.model.eeg_encoder.parameters():
            param.requires_grad = False
        for param in self.model.metadata_head.parameters():
            param.requires_grad = False
        
        # Unfreeze top 4 layers of BERT Decoder
        self.model.unfreeze_decoder_top_layers(num_layers=4)
        
        # Optimize adapter and trainable BERT layers
        params_to_optimize = list(self.model.adapter.parameters()) + \
                           [p for p in self.model.decoder.parameters() if p.requires_grad]
        
        optimizer = torch.optim.AdamW(params_to_optimize, lr=lr, weight_decay=0.01)
        
        total_steps = len(self.train_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
        )
        
        best_val_loss = float('inf')
        
        for epoch in range(1, epochs + 1):
            print(f"\nEpoch {epoch}/{epochs}")
            start_time = time.time()
            
            # Train
            self.model.train()
            total_loss = 0.0
            
            pbar = tqdm(self.train_loader, desc=f"Phase2 Epoch {epoch}")
            for eeg, _, text in pbar:
                eeg = eeg.to(self.device)
                text = text.to(self.device)
                
                labels = text[:, 1:].contiguous()
                input_ids = text[:, :-1].contiguous()
                
                optimizer.zero_grad()
                
                outputs = self.model(
                    eeg=eeg,
                    input_ids=input_ids,
                    labels=labels,
                    mode='generate'
                )
                
                loss = outputs['text_loss']
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(params_to_optimize, 1.0)
                optimizer.step()
                scheduler.step()
                
                total_loss += loss.item()
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
            # Validate
            val_loss = self._validate_generation()
            
            elapsed = time.time() - start_time
            avg_train_loss = total_loss / len(self.train_loader)
            
            print(f"Train Loss: {avg_train_loss:.4f}")
            print(f"Val Loss: {val_loss:.4f}")
            print(f"Time: {elapsed:.1f}s")
            
            self.history['phase2'].append({
                'epoch': epoch,
                'train_loss': avg_train_loss,
                'val_loss': val_loss
            })
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(self.model.state_dict(), self.output_dir / 'phase2_best.pt')
                print(f"✓ Saved best Phase 2 model (val_loss={best_val_loss:.4f})")
        
        self.model.load_state_dict(torch.load(self.output_dir / 'phase2_best.pt'))
        print("\nPhase 2 complete! Adapter can generate text.")

    @torch.no_grad()
    def _validate_generation(self):
        """Validate generation loss."""
        self.model.eval()
        total_loss = 0.0
        
        for eeg, _, text in self.val_loader:
            eeg = eeg.to(self.device)
            text = text.to(self.device)
            
            labels = text[:, 1:].contiguous()
            input_ids = text[:, :-1].contiguous()
            
            outputs = self.model(
                eeg=eeg,
                input_ids=input_ids,
                labels=labels,
                mode='generate'
            )
            
            total_loss += outputs['text_loss'].item()
        
        return total_loss / len(self.val_loader)
    
    def phase3_multitask_polishing(self, epochs=15, lr=5e-6, warmup_steps=500):
        print("\n" + "="*80)
        print("PHASE 3: Multitask Polishing (End-to-End)")
        print("="*80)
        
        # Unfreeze EEG encoder
        for param in self.model.eeg_encoder.parameters():
            param.requires_grad = True
        
        # Unfreeze all BERT Decoder
        self.model.unfreeze_decoder_all()
        
        # Differential learning rates
        param_groups = [
            {'params': self.model.decoder.parameters(), 'lr': lr}, # 5e-6
            {'params': self.model.eeg_encoder.parameters(), 'lr': lr * 5}, # 2.5e-5
            {'params': self.model.adapter.parameters(), 'lr': lr * 10}, # 5e-5
            {'params': self.model.metadata_head.parameters(), 'lr': lr * 10} # 5e-5
        ]
        
        optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)
        
        total_steps = len(self.train_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
        )
        
        # Loss weights
        text_weight = 1.0
        color_weight = 0.2
        object_weight = 0.3
        
        best_val_loss = float('inf')
        
        for epoch in range(1, epochs + 1):
            print(f"\nEpoch {epoch}/{epochs}")
            start_time = time.time()
            
            # Train
            self.model.train()
            total_loss, total_text_loss, total_color_loss, total_object_loss = 0, 0, 0, 0
            
            pbar = tqdm(self.train_loader, desc=f"Phase3 Epoch {epoch}")
            for eeg, metadata, text in pbar:
                eeg = eeg.to(self.device)
                metadata = metadata.to(self.device)
                text = text.to(self.device)
                
                labels = text[:, 1:].contiguous()
                input_ids = text[:, :-1].contiguous()
                
                optimizer.zero_grad()
                
                outputs = self.model(
                    eeg=eeg,
                    input_ids=input_ids,
                    labels=labels,
                    mode='multitask'
                )
                
                # Compute all losses
                text_loss = outputs['text_loss']
                
                color_targets = metadata[:, 0].long()
                color_loss = self.color_criterion(outputs['color_logits'], color_targets)
                
                object_targets = metadata[:, 1:].float()
                object_loss = self.object_criterion(outputs['object_logits'], object_targets)
                
                # Combined loss
                loss = text_weight * text_loss + \
                       color_weight * color_loss + \
                       object_weight * object_loss
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                
                total_loss += loss.item()
                total_text_loss += text_loss.item()
                total_color_loss += color_loss.item()
                total_object_loss += object_loss.item()
                
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'txt': f'{text_loss.item():.4f}',
                    'clr': f'{color_loss.item():.4f}',
                    'obj': f'{object_loss.item():.4f}'
                })
            
            # Validate
            val_metrics = self._validate_multitask(text_weight, color_weight, object_weight)
            
            elapsed = time.time() - start_time
            n = len(self.train_loader)
            
            print(f"Train - Loss: {total_loss/n:.4f} | "
                  f"Text: {total_text_loss/n:.4f} | "
                  f"Color: {total_color_loss/n:.4f} | "
                  f"Object: {total_object_loss/n:.4f}")
            print(f"Val - Loss: {val_metrics['loss']:.4f} | "
                  f"Text: {val_metrics['text_loss']:.4f} | "
                  f"Color: {val_metrics['color_loss']:.4f} | "
                  f"Object: {val_metrics['object_loss']:.4f}")
            print(f"Time: {elapsed:.1f}s | LR: {optimizer.param_groups[0]['lr']:.2e}")
            
            self.history['phase3'].append({
                'epoch': epoch,
                'train_loss': total_loss / n,
                'val_loss': val_metrics['loss'],
                'val_color_acc': val_metrics['color_accuracy'], # Added for logging
                'val_object_f1': val_metrics['object_f1_micro'], # Added for logging
            })
            
            if val_metrics['loss'] < best_val_loss:
                best_val_loss = val_metrics['loss']
                torch.save(self.model.state_dict(), self.output_dir / 'phase3_best.pt')
                torch.save(self.model.state_dict(), self.output_dir / 'final_model.pt')
                print(f"✓ Saved best Phase 3 model (val_loss={best_val_loss:.4f})")
        
        print("\nPhase 3 complete! Training finished.")
        
        with open(self.output_dir / 'training_history.json', 'w') as f:
            json.dump(self.history, f, indent=2)

    @torch.no_grad()
    def _validate_multitask(self, text_weight, color_weight, object_weight):
        """Validate multitask loss AND METRICS."""
        self.model.eval()
        total_loss, total_text_loss, total_color_loss, total_object_loss = 0, 0, 0, 0
        
        all_color_preds = []
        all_color_targets = []
        all_object_preds = []
        all_object_targets = []
        
        for eeg, metadata, text in self.val_loader:
            eeg = eeg.to(self.device)
            metadata = metadata.to(self.device)
            text = text.to(self.device)
            
            labels = text[:, 1:].contiguous()
            input_ids = text[:, :-1].contiguous()
            
            outputs = self.model(
                eeg=eeg,
                input_ids=input_ids,
                labels=labels,
                mode='multitask'
            )
            
            text_loss = outputs['text_loss']
            
            color_targets = metadata[:, 0].long()
            color_loss = self.color_criterion(outputs['color_logits'], color_targets)
            
            object_targets = metadata[:, 1:].float()
            object_loss = self.object_criterion(outputs['object_logits'], object_targets)
            
            loss = text_weight * text_loss + \
                   color_weight * color_loss + \
                   object_weight * object_loss
            
            total_loss += loss.item()
            total_text_loss += text_loss.item()
            total_color_loss += color_loss.item()
            total_object_loss += object_loss.item()
            
            all_color_preds.append(outputs['color_logits'].argmax(dim=-1).cpu())
            all_color_targets.append(color_targets.cpu())
            all_object_preds.append((torch.sigmoid(outputs['object_logits']) > 0.5).cpu())
            all_object_targets.append(object_targets.cpu())
        
        all_color_preds = np.concatenate(all_color_preds)
        all_color_targets = np.concatenate(all_color_targets)
        all_object_preds = np.concatenate(all_object_preds)
        all_object_targets = np.concatenate(all_object_targets)
        
        metrics = compute_metrics_numpy(
            all_color_preds,
            all_color_targets,
            all_object_preds,
            all_object_targets
        )
        
        n = len(self.val_loader)
        return {
            'loss': total_loss / n,
            'text_loss': total_text_loss / n,
            'color_loss': total_color_loss / n,
            'object_loss': total_object_loss / n,
            'color_accuracy': metrics['color_accuracy'], # For Phase 3 logging
            'object_f1_micro': metrics['object_f1_micro'] # For Phase 3 logging
        }


def main():
    """Main training script."""
    
    # Configuration
    H5_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
    BERT_MODEL = "/home/poorna/models/bert-base-uncased"
    CHECKPOINT_DIR = "./checkpoints"
    
    BATCH_SIZE = 16
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Using device: {DEVICE}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token # Or tokenizer.pad_token = "[PAD]"
    # --- Add BERT-specific tokens ---
    if tokenizer.bos_token is None:
        tokenizer.bos_token = tokenizer.cls_token
    if tokenizer.eos_token is None:
        tokenizer.eos_token = tokenizer.sep_token
    # --------------------------------
    
    # Create dataset
    dataset = EEGTextDataset(H5_PATH)
    n_train = int(len(dataset) * 0.8)
    n_val = int(len(dataset) * 0.1)
    n_test = len(dataset) - n_train - n_val
    
    train_ds, val_ds, test_ds = random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id),
        num_workers=0,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id),
        num_workers=0,
        pin_memory=True
    )
    
    # Create model
    model = EEGToTextModel(
        num_channels=62,
        time_steps=400,
        num_colors=12,
        num_objects=90,
        bert_model_name=BERT_MODEL,
        # --- THIS LINE IS NOW REMOVED ---
        # gpt2_model_name="gpt2", 
        # ---------------------------------
        eeg_encoder_dim=512,
        projection_dim=256,
        num_eeg_prefix_tokens=8
    ).to(DEVICE)
    
    # Create trainer
    trainer = ThreePhaseTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        tokenizer=tokenizer,
        device=DEVICE,
        output_dir=CHECKPOINT_DIR
    )
    
    # Run three-phase training
    print("\n" + "="*80)
    print("STARTING THREE-PHASE CONCEPT-THEN-LANGUAGE TRAINING")
    print("="*80)
    
    # --- Corrected function name ---
    trainer.phase1_concept_pretraining(epochs=15, lr=1e-3) 
    trainer.phase2_generative_finetuning(epochs=10, lr=1e-4)
    trainer.phase3_multitask_polishing(epochs=15, lr=5e-6)
    
    print("\n" + "="*80)
    print("TRAINING COMPLETE!")
    print("="*80)
    print(f"Final model saved to: {Path(CHECKPOINT_DIR) / 'final_model.pt'}")


if __name__ == "__main__":
    main()

/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


Using device: cuda
FIXED SimplifiedEEGEncoder: 62 channels -> 512d, over 400 steps
Loading BERT from /home/poorna/models/bert-base-uncased...
BERT text encoder frozen
Projection heads: 512d, 768d -> 256d
Loading BERT LM Head from /home/poorna/models/bert-base-uncased...
BERT Decoder weights frozen initially
EEGToAdapter: 512d -> 8 tokens -> 768d
Total parameters: 235,253,409
Trainable parameters: 16,256,871

STARTING THREE-PHASE CONCEPT-THEN-LANGUAGE TRAINING

PHASE 1: Concept Pre-training (Training EEG-Encoder + Metadata-Heads)
Phase 1 Loss Weights: Color=1.0, Object=100.0

Epoch 1/15


Phase1 Epoch 1:   0%|          | 0/1400 [00:00<?, ?it/s]

Train - Loss: 3.0019 | Color: 2.2353 | Object: 0.0077
Val - Loss: 2.9005 | Color Acc: 0.2389 (Baseline: ~0.083) | Object F1: 0.0000 (Baseline: 0.0)
Time: 191.8s
✓ Saved best Phase 1 model (val_loss=2.9005)

Epoch 2/15


Phase1 Epoch 2:   0%|          | 0/1400 [00:00<?, ?it/s]

Train - Loss: 2.9550 | Color: 2.2164 | Object: 0.0074
Val - Loss: 2.8973 | Color Acc: 0.2389 (Baseline: ~0.083) | Object F1: 0.0000 (Baseline: 0.0)
Time: 192.2s
✓ Saved best Phase 1 model (val_loss=2.8973)

Epoch 3/15


Phase1 Epoch 3:   0%|          | 0/1400 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
"""
Comprehensive evaluation script for EEG-to-Text model.
Computes BLEU, ROUGE, METEOR and metadata metrics.
"""

import torch
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split # Added random_split
from torch.nn.utils.rnn import pad_sequence # Added pad_sequence
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import evaluate as hf_evaluate
import json
from pathlib import Path

# We must import the model definition from Cell 1
# from models import EEGToTextModel # This will fail. We'll assume models.py exists

# =========================================================================
# --- Re-defining collate_fn here for the evaluator ---
# =========================================================================
def collate_fn_eval(batch, pad_id):
    """Collate batch with padding."""
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, text in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(text)
    
    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=pad_id)
    
    return eeg_batch, meta_batch, text_padded


class EEGTextDataset(Dataset):
    """Dataset for evaluation."""
    
    def __init__(self, h5_path, indices=None):
        self.h5_path = h5_path
        self.h5_file = None
        self.indices = indices
        
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]
        
        if self.indices is None:
            self.indices = list(range(self.n_samples))
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        real_idx = self.indices[idx]
        eeg = torch.from_numpy(self.h5_file['eeg'][real_idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][real_idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][real_idx].astype(np.int64))
        
        return eeg, meta, text


class EEGToTextEvaluator:
    """Comprehensive evaluator."""
    
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        
        # Load metrics
        print("Loading evaluation metrics...")
        self.bleu_metric = hf_evaluate.load('bleu')
        self.rouge_metric = hf_evaluate.load('rouge')
        self.meteor_metric = hf_evaluate.load('meteor')
        print("Metrics loaded successfully")
    
    @torch.no_grad()
    def evaluate_full(
        self,
        dataloader,
        max_length=50,
        num_beams=4,
        temperature=1.0,
        top_k=50,
        top_p=0.95
    ):
        """Generate and evaluate on full dataset."""
        self.model.eval()
        
        all_predictions = []
        all_references = []
        all_color_preds = []
        all_color_targets = []
        all_object_preds = []
        all_object_targets = []
        
        print("\nGenerating predictions...")
        for eeg_batch, meta_batch, text_batch in tqdm(dataloader):
            eeg_batch = eeg_batch.to(self.device)
            meta_batch = meta_batch.to(self.device)
            
            # Generate text
            # This now uses the *FIXED* generate function from the model
            generated_texts = self.model.generate(
                eeg=eeg_batch,
                tokenizer=self.tokenizer,
                max_length=max_length,
                num_beams=num_beams,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p
            )
            
            # Get reference texts
            reference_texts = self.tokenizer.batch_decode(
                text_batch,
                skip_special_tokens=True
            )
            
            # Get metadata predictions
            # We need to run a forward pass
            dummy_input_ids = torch.full(
                (eeg_batch.shape[0], 1), 
                self.tokenizer.bos_token_id, 
                dtype=torch.long, 
                device=self.device
            )
            outputs = self.model(
                eeg=eeg_batch,
                input_ids=dummy_input_ids,
                mode='multitask' # or 'generate', just needs return_metadata=True
            )
            color_logits = outputs['color_logits']
            object_logits = outputs['object_logits']

            color_preds = color_logits.argmax(dim=-1).cpu().numpy()
            object_preds = (torch.sigmoid(object_logits) > 0.5).cpu().numpy()
            
            # Store results
            all_predictions.extend(generated_texts)
            all_references.extend(reference_texts)
            all_color_preds.extend(color_preds)
            all_color_targets.extend(meta_batch[:, 0].cpu().numpy())
            all_object_preds.extend(object_preds)
            all_object_targets.extend(meta_batch[:, 1:].cpu().numpy())
        
        print("Computing metrics...")
        
        # Text metrics
        text_metrics = self.compute_text_metrics(all_predictions, all_references)
        
        # Metadata metrics
        metadata_metrics = self.compute_metadata_metrics(
            all_color_preds,
            all_color_targets,
            all_object_preds,
            all_object_targets
        )
        
        # Combine results
        results = {
            'text_metrics': text_metrics,
            'metadata_metrics': metadata_metrics,
            'num_samples': len(all_predictions),
            'examples': []
        }
        
        # Sample examples
        for i in range(min(10, len(all_predictions))):
            results['examples'].append({
                'reference': all_references[i],
                'prediction': all_predictions[i],
                'color_target': int(all_color_targets[i]),
                'color_pred': int(all_color_preds[i]),
            })
        
        return results
    
    def compute_text_metrics(self, predictions, references):
        """Compute BLEU, ROUGE, METEOR."""
        
        # Format references
        references_list = [[ref] for ref in references]
        
        # BLEU
        bleu_results = self.bleu_metric.compute(
            predictions=predictions,
            references=references_list,
            max_order=4
        )
        
        # ROUGE
        rouge_results = self.rouge_metric.compute(
            predictions=predictions,
            references=references
        )
        
        # METEOR
        meteor_results = self.meteor_metric.compute(
            predictions=predictions,
            references=references
        )
        
        return {
            'bleu': bleu_results['bleu'],
            'bleu_1': bleu_results['precisions'][0],
            'bleu_2': bleu_results['precisions'][1],
            'bleu_3': bleu_results['precisions'][2],
            'bleu_4': bleu_results['precisions'][3],
            'rouge1': rouge_results['rouge1'],
            'rouge2': rouge_results['rouge2'],
            'rougeL': rouge_results['rougeL'],
            'meteor': meteor_results['meteor']
        }
    
    def compute_metadata_metrics(
        self,
        color_preds,
        color_targets,
        object_preds,
        object_targets
    ):
        """Compute accuracy and F1 scores."""
        color_preds = np.array(color_preds)
        color_targets = np.array(color_targets)
        object_preds = np.array(object_preds)
        object_targets = np.array(object_targets)
        
        # Color accuracy
        color_acc = (color_preds == color_targets).mean()
        
        # Object metrics
        tp = (object_preds * object_targets).sum()
        fp = (object_preds * (1 - object_targets)).sum()
        fn = ((1 - object_preds) * object_targets).sum()
        
        # Micro F1
        precision = tp / (tp + fp + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        f1_micro = 2 * precision * recall / (precision + recall + 1e-10)
        
        # Macro F1
        f1_per_class = []
        for i in range(object_preds.shape[1]):
            tp_i = (object_preds[:, i] * object_targets[:, i]).sum()
            fp_i = (object_preds[:, i] * (1 - object_targets[:, i])).sum()
            fn_i = ((1 - object_preds[:, i]) * object_targets[:, i]).sum()
            
            prec_i = tp_i / (tp_i + fp_i + 1e-10)
            rec_i = tp_i / (tp_i + fn_i + 1e-10)
            f1_i = 2 * prec_i * rec_i / (prec_i + rec_i + 1e-10)
            f1_per_class.append(f1_i)
        
        f1_macro = np.mean(f1_per_class)
        
        return {
            'color_accuracy': float(color_acc),
            'object_f1_micro': float(f1_micro),
            'object_f1_macro': float(f1_macro),
            'object_precision': float(precision),
            'object_recall': float(recall)
        }
    
    def print_results(self, results):
        """Pretty print results."""
        print("\n" + "="*80)
        print("EVALUATION RESULTS")
        print("="*80)
        
        print(f"\nSamples evaluated: {results['num_samples']}")
        
        print("\n--- Text Generation Metrics ---")
        tm = results['text_metrics']
        print(f"BLEU-1: {tm['bleu_1']:.4f}")
        print(f"BLEU-2: {tm['bleu_2']:.4f}")
        print(f"BLEU-3: {tm['bleu_3']:.4f}")
        print(f"BLEU-4: {tm['bleu_4']:.4f}")
        print(f"BLEU (overall): {tm['bleu']:.4f}")
        print(f"\nROUGE-1: {tm['rouge1']:.4f}")
        print(f"ROUGE-2: {tm['rouge2']:.4f}")
        print(f"ROUGE-L: {tm['rougeL']:.4f}")
        print(f"\nMETEOR: {tm['meteor']:.4f}")
        
        print("\n--- Metadata Prediction Metrics ---")
        mm = results['metadata_metrics']
        print(f"Color Accuracy: {mm['color_accuracy']:.4f}")
        print(f"Object F1 (micro): {mm['object_f1_micro']:.4f}")
        print(f"Object F1 (macro): {mm['object_f1_macro']:.4f}")
        print(f"Object Precision: {mm['object_precision']:.4f}")
        print(f"Object Recall: {mm['object_recall']:.4f}")
        
        print("\n--- Sample Predictions ---")
        for i, ex in enumerate(results['examples'][:5]):
            print(f"\nExample {i+1}:\n")
            print(f"  Reference: {ex['reference']}\n")
            print(f"  Prediction: {ex['prediction']}\n")
            print(f"  Color (target/pred): {ex['color_target']} / {ex['color_pred']}\n")
        
        print("\n" + "="*80)


def main():
    """Main evaluation script."""
    
    # Configuration
    H5_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
    BERT_MODEL = "/home/poorna/models/bert-base-uncased"
    CHECKPOINT_PATH = "./checkpoints/final_model.pt"
    OUTPUT_PATH = "./checkpoints/evaluation_results.json"
    
    BATCH_SIZE = 16
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Using device: {DEVICE}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Create test dataset (last 10%)
    with h5py.File(H5_PATH, 'r') as f:
        n_total = f['eeg'].shape[0]
    
    n_train = int(n_total * 0.8)
    n_val = int(n_total * 0.1)
    test_indices = list(range(n_train + n_val, n_total))
    
    test_dataset = EEGTextDataset(H5_PATH, indices=test_indices)
    
    # =========================================================================
    # --- FIX 2: Added collate_fn and set num_workers=0 ---
    # =========================================================================
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda b: collate_fn_eval(b, tokenizer.pad_token_id),
        num_workers=0 
    )
    
    print(f"Test set size: {len(test_dataset)} samples")
    
    # Load model
    print(f"\nLoading model from {CHECKPOINT_PATH}")
    model = EEGToTextModel(
        num_channels=62,
        time_steps=400,
        num_colors=12,
        num_objects=90,
        bert_model_name=BERT_MODEL,
        gpt2_model_name="gpt2",
        eeg_encoder_dim=512,
        projection_dim=256,
        num_eeg_prefix_tokens=8
    ).to(DEVICE)
    
    try:
        model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
        print("Model loaded successfully")
    except FileNotFoundError:
        print(f"ERROR: Checkpoint file not found at {CHECKPOINT_PATH}")
        print("Please run the training script first.")
        return
    except Exception as e:
        print(f"ERROR: Failed to load model. Mismatched keys or architecture?")
        print(e)
        return
        
    # Create evaluator
    evaluator = EEGToTextEvaluator(model, tokenizer, DEVICE)
    
    # Evaluate with beam search
    print("\n" + "="*80)
    print("BEAM SEARCH EVALUATION (num_beams=4)")
    print("="*80)
    
    results = evaluator.evaluate_full(
        test_loader,
        max_length=50,
        num_beams=4
    )
    
    evaluator.print_results(results)
    
    # Save results
    with open(OUTPUT_PATH, 'w') as f:
        json.dump(results, f, indent=2, default=float)
    
    print(f"\nResults saved to {OUTPUT_PATH}")
    

if __name__ == "__main__":
    main()

Using device: cuda
Test set size: 2800 samples

Loading model from ./checkpoints/final_model.pt
SimplifiedEEGEncoder: 62 channels -> 512d, 4 layers
Loading BERT from /home/poorna/models/bert-base-uncased...
BERT text encoder frozen
Projection heads: 512d, 768d -> 256d
GPT-2 weights frozen initially
EEGToGPT2Adapter: 512d -> 8 tokens -> 768d
Total parameters: 250,178,919
Trainable parameters: 16,256,871
ERROR: Failed to load model. Mismatched keys or architecture?
CUDA out of memory. Tried to allocate 148.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 40.50 MiB is free. Process 10286 has 4.90 GiB memory in use. Including non-PyTorch memory, this process has 2.65 GiB memory in use. Of the allocated memory 2.37 GiB is allocated by PyTorch, and 177.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.o

In [6]:
"""
Comprehensive evaluation script for EEG-to-Text model.
Computes BLEU, ROUGE, METEOR and metadata metrics.
"""

import torch
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split # Added random_split
from torch.nn.utils.rnn import pad_sequence # Added pad_sequence
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import evaluate as hf_evaluate
import json
from pathlib import Path

# We must import the model definition from Cell 1
# from models import EEGToTextModel # This will fail. We'll assume models.py exists

# =========================================================================
# --- Re-defining collate_fn here for the evaluator ---
# =========================================================================
def collate_fn_eval(batch, pad_id):
    """Collate batch with padding."""
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, text in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(text)
    
    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=pad_id)
    
    return eeg_batch, meta_batch, text_padded


class EEGTextDataset(Dataset):
    """Dataset for evaluation."""
    
    def __init__(self, h5_path, indices=None):
        self.h5_path = h5_path
        self.h5_file = None
        self.indices = indices
        
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]
        
        if self.indices is None:
            self.indices = list(range(self.n_samples))
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        real_idx = self.indices[idx]
        eeg = torch.from_numpy(self.h5_file['eeg'][real_idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][real_idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][real_idx].astype(np.int64))
        
        return eeg, meta, text


class EEGToTextEvaluator:
    """Comprehensive evaluator."""
    
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        
        # Load metrics
        print("Loading evaluation metrics...")
        self.bleu_metric = hf_evaluate.load('bleu')
        self.rouge_metric = hf_evaluate.load('rouge')
        self.meteor_metric = hf_evaluate.load('meteor')
        print("Metrics loaded successfully")
    
    @torch.no_grad()
    def evaluate_full(
        self,
        dataloader,
        max_length=50,
        num_beams=4,
        temperature=1.0,
        top_k=50,
        top_p=0.95
    ):
        """Generate and evaluate on full dataset."""
        self.model.eval()
        
        all_predictions = []
        all_references = []
        all_color_preds = []
        all_color_targets = []
        all_object_preds = []
        all_object_targets = []
        
        # =========================================================================
        # --- FIX: Determine the correct start token ID once ---
        # =========================================================================
        self.start_token_id = self.tokenizer.bos_token_id \
            if self.tokenizer.bos_token_id is not None \
            else self.tokenizer.cls_token_id
        # =========================================================================

        print("\nGenerating predictions...")
        for eeg_batch, meta_batch, text_batch in tqdm(dataloader):
            eeg_batch = eeg_batch.to(self.device)
            meta_batch = meta_batch.to(self.device)
            
            # Generate text
            generated_texts = self.model.generate(
                eeg=eeg_batch,
                tokenizer=self.tokenizer,
                max_length=max_length,
                num_beams=num_beams,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p
            )
            
            # Get reference texts
            reference_texts = self.tokenizer.batch_decode(
                text_batch,
                skip_special_tokens=True
            )
            
            # Get metadata predictions
            # We need to run a forward pass
            
            # =========================================================================
            # --- FIX: Use the determined start_token_id ---
            # =========================================================================
            dummy_input_ids = torch.full(
                (eeg_batch.shape[0], 1), 
                self.start_token_id,  # <-- Was: self.tokenizer.bos_token_id
                dtype=torch.long, 
                device=self.device
            )
            # =========================================================================
            
            outputs = self.model(
                eeg=eeg_batch,
                input_ids=dummy_input_ids,
                mode='multitask' # or 'generate', just needs return_metadata=True
            )
            color_logits = outputs['color_logits']
            object_logits = outputs['object_logits']

            color_preds = color_logits.argmax(dim=-1).cpu().numpy()
            object_preds = (torch.sigmoid(object_logits) > 0.5).cpu().numpy()
            
            # Store results
            all_predictions.extend(generated_texts)
            all_references.extend(reference_texts)
            all_color_preds.extend(color_preds)
            all_color_targets.extend(meta_batch[:, 0].cpu().numpy())
            all_object_preds.extend(object_preds)
            all_object_targets.extend(meta_batch[:, 1:].cpu().numpy())
        
        print("Computing metrics...")
        
        # Text metrics
        text_metrics = self.compute_text_metrics(all_predictions, all_references)
        
        # Metadata metrics
        metadata_metrics = self.compute_metadata_metrics(
            all_color_preds,
            all_color_targets,
            all_object_preds,
            all_object_targets
        )
        
        # Combine results
        results = {
            'text_metrics': text_metrics,
            'metadata_metrics': metadata_metrics,
            'num_samples': len(all_predictions),
            'examples': []
        }
        
        # Sample examples
        for i in range(min(10, len(all_predictions))):
            results['examples'].append({
                'reference': all_references[i],
                'prediction': all_predictions[i],
                'color_target': int(all_color_targets[i]),
                'color_pred': int(all_color_preds[i]),
            })
        
        return results
    
    def compute_text_metrics(self, predictions, references):
        """Compute BLEU, ROUGE, METEOR."""
        
        # Format references
        references_list = [[ref] for ref in references]
        
        # BLEU
        bleu_results = self.bleu_metric.compute(
            predictions=predictions,
            references=references_list,
            max_order=4
        )
        
        # ROUGE
        rouge_results = self.rouge_metric.compute(
            predictions=predictions,
            references=references
        )
        
        # METEOR
        meteor_results = self.meteor_metric.compute(
            predictions=predictions,
            references=references
        )
        
        return {
            'bleu': bleu_results['bleu'],
            'bleu_1': bleu_results['precisions'][0],
            'bleu_2': bleu_results['precisions'][1],
            'bleu_3': bleu_results['precisions'][2],
            'bleu_4': bleu_results['precisions'][3],
            'rouge1': rouge_results['rouge1'],
            'rouge2': rouge_results['rouge2'],
            'rougeL': rouge_results['rougeL'],
            'meteor': meteor_results['meteor']
        }
    
    def compute_metadata_metrics(
        self,
        color_preds,
        color_targets,
        object_preds,
        object_targets
    ):
        """Compute accuracy and F1 scores."""
        color_preds = np.array(color_preds)
        color_targets = np.array(color_targets)
        object_preds = np.array(object_preds)
        object_targets = np.array(object_targets)
        
        # Color accuracy
        color_acc = (color_preds == color_targets).mean()
        
        # Object metrics
        tp = (object_preds * object_targets).sum()
        fp = (object_preds * (1 - object_targets)).sum()
        fn = ((1 - object_preds) * object_targets).sum()
        
        # Micro F1
        precision = tp / (tp + fp + 1e-10)
        recall = tp / (tp + fn + 1e-10)
        f1_micro = 2 * precision * recall / (precision + recall + 1e-10)
        
        # Macro F1
        f1_per_class = []
        for i in range(object_preds.shape[1]):
            tp_i = (object_preds[:, i] * object_targets[:, i]).sum()
            fp_i = (object_preds[:, i] * (1 - object_targets[:, i])).sum()
            fn_i = ((1 - object_preds[:, i]) * object_targets[:, i]).sum()
            
            prec_i = tp_i / (tp_i + fp_i + 1e-10)
            rec_i = tp_i / (tp_i + fn_i + 1e-10)
            f1_i = 2 * prec_i * rec_i / (prec_i + rec_i + 1e-10)
            f1_per_class.append(f1_i)
        
        f1_macro = np.mean(f1_per_class)
        
        return {
            'color_accuracy': float(color_acc),
            'object_f1_micro': float(f1_micro),
            'object_f1_macro': float(f1_macro),
            'object_precision': float(precision),
            'object_recall': float(recall)
        }
    
    def print_results(self, results):
        """Pretty print results."""
        print("\n" + "="*80)
        print("EVALUATION RESULTS")
        print("="*80)
        
        print(f"\nSamples evaluated: {results['num_samples']}")
        
        print("\n--- Text Generation Metrics ---")
        tm = results['text_metrics']
        print(f"BLEU-1: {tm['bleu_1']:.4f}")
        print(f"BLEU-2: {tm['bleu_2']:.4f}")
        print(f"BLEU-3: {tm['bleu_3']:.4f}")
        print(f"BLEU-4: {tm['bleu_4']:.4f}")
        print(f"BLEU (overall): {tm['bleu']:.4f}")
        print(f"\nROUGE-1: {tm['rouge1']:.4f}")
        print(f"ROUGE-2: {tm['rouge2']:.4f}")
        print(f"ROUGE-L: {tm['rougeL']:.4f}")
        print(f"\nMETEOR: {tm['meteor']:.4f}")
        
        print("\n--- Metadata Prediction Metrics ---")
        mm = results['metadata_metrics']
        print(f"Color Accuracy: {mm['color_accuracy']:.4f}")
        print(f"Object F1 (micro): {mm['object_f1_micro']:.4f}")
        print(f"Object FF (macro): {mm['object_f1_macro']:.4f}")
        print(f"Object Precision: {mm['object_precision']:.4f}")
        print(f"Object Recall: {mm['object_recall']:.4f}")
        
        print("\n--- Sample Predictions ---")
        for i, ex in enumerate(results['examples'][:5]):
            print(f"\nExample {i+1}:\n")
            print(f"  Reference: {ex['reference']}\n")
            print(f"  Prediction: {ex['prediction']}\n")
            print(f"  Color (target/pred): {ex['color_target']} / {ex['color_pred']}\n")
        
        print("\n" + "="*80)


def main():
    """Main evaluation script."""
    
    # Configuration
    H5_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
    BERT_MODEL = "/home/poorna/models/bert-base-uncased"
    CHECKPOINT_PATH = "./checkpoints/final_model.pt"
    OUTPUT_PATH = "./checkpoints/evaluation_results.json"
    
    BATCH_SIZE = 16
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Using device: {DEVICE}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Create test dataset (last 10%)
    with h5py.File(H5_PATH, 'r') as f:
        n_total = f['eeg'].shape[0]
    
    n_train = int(n_total * 0.8)
    n_val = int(n_total * 0.1)
    test_indices = list(range(n_train + n_val, n_total))
    
    test_dataset = EEGTextDataset(H5_PATH, indices=test_indices)
    
    # =========================================================================
    # --- FIX 2: Added collate_fn and set num_workers=0 ---
    # =========================================================================
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda b: collate_fn_eval(b, tokenizer.pad_token_id),
        num_workers=0 
    )
    
    print(f"Test set size: {len(test_dataset)} samples")
    
    # Load model
    print(f"\nLoading model from {CHECKPOINT_PATH}")
    model = EEGToTextModel(
        num_channels=62,
        time_steps=400,
        num_colors=12,
        num_objects=90,
        bert_model_name=BERT_MODEL,
        gpt2_model_name="gpt2",
        eeg_encoder_dim=512,
        projection_dim=256,
        num_eeg_prefix_tokens=8
    ).to(DEVICE)
    
    try:
        model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
        print("Model loaded successfully")
    except FileNotFoundError:
        print(f"ERROR: Checkpoint file not found at {CHECKPOINT_PATH}")
        print("Please run the training script first.")
        return
    except Exception as e:
        print(f"ERROR: Failed to load model. Mismatched keys or architecture?")
        print(e)
        return
        
    # Create evaluator
    evaluator = EEGToTextEvaluator(model, tokenizer, DEVICE)
    
    # Evaluate with beam search
    print("\n" + "="*80)
    print("BEAM SEARCH EVALUATION (num_beams=4)")
    print("="*80)
    
    results = evaluator.evaluate_full(
        test_loader,
        max_length=50,
        num_beams=4
    )
    
    evaluator.print_results(results)
    
    # Save results
    with open(OUTPUT_PATH, 'w') as f:
        json.dump(results, f, indent=2, default=float)
    
    print(f"\nResults saved to {OUTPUT_PATH}")
    

if __name__ == "__main__":
    main()

Using device: cuda
Test set size: 2800 samples

Loading model from ./checkpoints/final_model.pt
SimplifiedEEGEncoder: 62 channels -> 512d, 4 layers
Loading BERT from /home/poorna/models/bert-base-uncased...
BERT text encoder frozen
Projection heads: 512d, 768d -> 256d


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


GPT-2 weights frozen initially
EEGToGPT2Adapter: 512d -> 8 tokens -> 768d
Total parameters: 250,178,919
Trainable parameters: 16,256,871
Model loaded successfully
Loading evaluation metrics...
Metrics loaded successfully

BEAM SEARCH EVALUATION (num_beams=4)

Generating predictions...


[nltk_data] Downloading package wordnet to /home/poorna/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/poorna/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/poorna/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


  0%|          | 0/175 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=

Computing metrics...

EVALUATION RESULTS

Samples evaluated: 2800

--- Text Generation Metrics ---
BLEU-1: 0.1235
BLEU-2: 0.0000
BLEU-3: 0.0000
BLEU-4: 0.0000
BLEU (overall): 0.0000

ROUGE-1: 0.0573
ROUGE-2: 0.0000
ROUGE-L: 0.0571

METEOR: 0.0199

--- Metadata Prediction Metrics ---
Color Accuracy: 0.2400
Object F1 (micro): 0.0000
Object FF (macro): 0.0000
Object Precision: 0.0000
Object Recall: 0.0000

--- Sample Predictions ---

Example 1:

  Reference: a bustling city street with tall buildings and moving vehicles.

  Prediction: handsd coffee in background

  Color (target/pred): 4 / 4


Example 2:

  Reference: a city street at dusk, illuminated by streetlights and vehicle headlights.

  Prediction: handsd coffee in background

  Color (target/pred): 4 / 4


Example 3:

  Reference: aerial view of a modern cityscape with diverse buildings and green spaces.

  Prediction: handsd coffee in background

  Color (target/pred): 4 / 4


Example 4:

  Reference: a bustling cityscape with 